# RICONOSCIMENTO DI ANIMALI PER AUTO A GUIDA AUTONOMA

**VisionTech Solutions** vuole sviluppare un sistema di riconoscimento automatico delle immagini
per distinguere tra **veicoli** e **animali**. L'obiettivo è ottimizzare il monitoraggio della fauna
nelle aree urbane, prevenendo incidenti stradali e migliorando la sicurezza cittadina.

Il sistema sfrutterà una **rete neurale convoluzionale (CNN)** addestrata sul dataset CIFAR-10,
che contiene immagini etichettate in 10 categorie, tra cui animali e veicoli.

### Obiettivi Specifici
- Classificare le immagini come **veicolo** o **animale** (classificazione binaria)
- **Massimizzare il Recall sugli animali**: in un sistema di guida autonoma, un animale
  non rilevato (falso negativo) è molto più pericoloso di un falso allarme (falso positivo)
- Analizzare errori, confusioni e potenziali miglioramenti tramite esperimenti sistematici

## PERCHÉ MASSIMIZZARE IL RECALL E NON L'ACCURACY?

In questo problema il costo degli errori **non è simmetrico**:

| Errore | Conseguenza |
|--------|------------|
| **Falso Negativo** (animale → non rilevato) | L'auto non frena → possibile incidente |
| **Falso Positivo** (veicolo → classificato come animale) | L'auto rallenta inutilmente → solo fastidio |

Per questo motivo la **metrica primaria è il Recall della classe 'Animale'**:
```
Recall = TP / (TP + FN)
```
Vogliamo che il modello identifichi quanti più animali possibile, accettando
qualche falso positivo in cambio di pochi falsi negativi.

## SEZIONE 1: IMPORTAZIONI E CONFIGURAZIONE

In [ ]:
# ============================================================
# LIBRERIE E MOTIVAZIONE DELLE SCELTE
# ============================================================

import torch                          # Framework Deep Learning principale
import torch.nn as nn                 # Moduli per costruire reti neurali (layer, loss)
import torch.optim as optim           # Ottimizzatori (Adam, SGD)
import torch.nn.functional as F       # Funzioni di attivazione e loss stateless
from torch.utils.data import DataLoader, random_split, Subset  # Gestione dataset

import torchvision                    # Dataset e trasformazioni per immagini
from torchvision import datasets, transforms  # CIFAR-10 e pipeline di preprocessing

# albumentations: libreria di data augmentation avanzata, più flessibile di torchvision.transforms
# Motivazione: offre trasformazioni fotograficamente realistiche (blur, brightness,
# rotazione) che simulano le condizioni reali in cui opererebbe il sistema di guida autonoma
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split  # Split stratificato train/val
from sklearn.metrics import classification_report, confusion_matrix  # Metriche

import matplotlib.pyplot as plt       # Visualizzazione grafici
import seaborn as sns                 # Heatmap per la confusion matrix
import numpy as np                    # Operazioni su array
import random                         # Seed per riproducibilità
import os                             # Gestione path e directory
import gc                             # Garbage collector: libera RAM GPU dopo ogni esperimento
                                      # Evita 'Session crashed due to out-of-memory' su Colab

## SEZIONE 2: SEED E DEVICE

Il seed garantisce la **riproducibilità** degli esperimenti: eseguendo il notebook
più volte si ottengono sempre gli stessi risultati, rendendo i confronti tra
esperimenti affidabili e non dipendenti dalla casualità.

In [ ]:
# ============================================================
# SEED PER RIPRODUCIBILITÀ
# ============================================================
# Impostiamo il seed su tutti i generatori casuali usati:
# - random: libreria standard Python
# - numpy: operazioni numeriche
# - torch: operazioni su tensori (CPU)
# - torch.cuda: operazioni su GPU
# - PYTHONHASHSEED: hashing Python (influenza dict, set)
# - cudnn.deterministic: forza algoritmi deterministici su GPU (più lento ma riproducibile)

SEED = 42

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# deterministic=True: forza CUDA a usare algoritmi deterministici (riproducibilità garantita)
# benchmark=False: disabilita la selezione automatica dell'algoritmo più veloce
#   (che è non deterministica). Trade-off: leggermente più lento ma riproducibile.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# ============================================================
# SELEZIONE DEL DEVICE
# ============================================================
# Le CNN richiedono molti calcoli matriciali: la GPU li esegue in parallelo
# su migliaia di core, accelerando il training di 10-100x rispetto alla CPU.
# Su Google Colab è disponibile una GPU NVIDIA (Tesla T4 o simile) gratuitamente.

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

if device.type == "cuda":
    print(f"GPU rilevata: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU disponibile: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## SEZIONE 3: DATASET CIFAR-10 — CARICAMENTO E FILTRAGGIO

**CIFAR-10** contiene 60.000 immagini RGB 32×32 pixel in 10 classi bilanciate (6.000 per classe):
- 50.000 immagini di training
- 10.000 immagini di test

**Classi disponibili:**

| ID | Nome | Categoria nel progetto |
|----|------|------------------------|
| 0 | airplane | scartata |
| 1 | automobile | **Veicolo** |
| 2 | bird | **Animale** |
| 3 | cat | **Animale** |
| 4 | deer | **Animale** |
| 5 | dog | **Animale** |
| 6 | frog | **Animale** |
| 7 | horse | **Animale** |
| 8 | ship | scartata |
| 9 | truck | **Veicolo** |

**Perché scartare airplane e ship?**  
Il progetto riguarda la sicurezza **stradale urbana**: aerei e navi non sono
rilevanti per un sistema di guida autonoma su strada. Includerli introdurrebbe
rumore e complicherebbe inutilmente il problema.

In [ ]:
# ============================================================
# TRASFORMAZIONE BASE (senza augmentation)
# ============================================================
# Usata per il test set e per i primi esperimenti.
#
# ToTensor(): converte l'immagine PIL [0,255] in tensore PyTorch [0,1]
# Normalize((0.5,), (0.5,)): porta i valori in [-1, 1]
#   Formula: (pixel - 0.5) / 0.5
#   Motivazione: i pesi iniziali della rete (Xavier/Kaiming) sono centrati in 0;
#   normalizzare l'input nello stesso range [-1,1] accelera la convergenza e
#   migliora la stabilità numerica del gradiente.

transform_base = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # per tutti e 3 i canali RGB
])

In [ ]:
# ============================================================
# DOWNLOAD DEL DATASET
# ============================================================
# torchvision.datasets.CIFAR10 scarica automaticamente il dataset
# nella cartella ./data al primo avvio (poi lo riutilizza dalla cache).
# Funziona sia in locale che su Google Colab senza importazioni speciali.

trainset_full = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_base
)
testset_full = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_base
)

print("Classi CIFAR-10:")
for idx, class_name in enumerate(trainset_full.classes):
    print(f"  {idx}: {class_name}")

In [ ]:
# ============================================================
# DEFINIZIONE DELLE CLASSI RILEVANTI
# ============================================================
# Selezioniamo solo le classi pertinenti al problema di guida autonoma stradale:
# - Animali (label → 0): bird, cat, deer, dog, frog, horse
# - Veicoli stradali (label → 1): automobile, truck
# Escluse: airplane (0), ship (8) → non rilevanti per strade urbane

animal_classes = [2, 3, 4, 5, 6, 7]   # bird, cat, deer, dog, frog, horse
vehicle_road_classes = [1, 9]          # automobile, truck

print(f"Classi animale: {[trainset_full.classes[i] for i in animal_classes]}")
print(f"Classi veicolo: {[trainset_full.classes[i] for i in vehicle_road_classes]}")


def binary_label(label):
    """Converte l'etichetta CIFAR-10 in etichetta binaria: 0=animale, 1=veicolo."""
    return 0 if label in animal_classes else 1

In [ ]:
# ============================================================
# DATASET PERSONALIZZATO: BinaryCIFAR10
# ============================================================
# Creiamo un Dataset PyTorch personalizzato che:
# 1. Filtra il CIFAR-10 originale tenendo solo animali e veicoli stradali
# 2. Rimappa le etichette in binario (0=animale, 1=veicolo)
#
# Perché una classe Dataset personalizzata?
# PyTorch richiede che ogni Dataset implementi __getitem__ e __len__.
# Ereditare da torch.utils.data.Dataset garantisce compatibilità con
# DataLoader, Subset, e tutte le utility di PyTorch.

class BinaryCIFAR10(torch.utils.data.Dataset):
    """
    Filtra CIFAR-10 per contenere solo animali e veicoli stradali,
    rimappando le etichette in binario: 0=animale, 1=veicolo.
    """
    def __init__(self, cifar_dataset):
        self.data = []
        self.targets = []

        # Itera su tutto il dataset CIFAR-10 originale
        # e mantieni solo i campioni delle classi rilevanti
        for img, label in cifar_dataset:
            if label in animal_classes + vehicle_road_classes:
                self.data.append(img)                    # tensore [3, 32, 32]
                self.targets.append(binary_label(label)) # 0 o 1

    def __getitem__(self, index):
        return self.data[index], self.targets[index]

    def __len__(self):
        return len(self.targets)


# Costruiamo i dataset binari
# NOTA: il test_dataset viene costruito qui e NON viene mai toccato durante
# il training/validation: è riservato alla valutazione finale del modello
binary_train_dataset = BinaryCIFAR10(trainset_full)
test_dataset = BinaryCIFAR10(testset_full)

print(f"Training set binario: {len(binary_train_dataset)} campioni")
print(f"Test set binario:     {len(test_dataset)} campioni")

n_animali = sum(1 for t in binary_train_dataset.targets if t == 0)
n_veicoli = sum(1 for t in binary_train_dataset.targets if t == 1)
print(f"  Animali (0): {n_animali} | Veicoli (1): {n_veicoli}")

## SEZIONE 4: SUDDIVISIONE STRATIFICATA TRAIN / VALIDATION

### Perché `train_test_split` con `stratify` invece di `random_split`?

- **`random_split`** suddivide casualmente senza garantire la proporzione delle classi
  nei due set. Con dataset sbilanciati (animali 6:2 rispetto ai veicoli) rischieremmo
  di avere troppi o troppo pochi campioni di una classe nella validation.
- **`train_test_split` con `stratify`** garantisce che la proporzione delle classi
  (animali vs veicoli) sia la stessa nel training set e nel validation set.
  Questo rende la validation più rappresentativa e le metriche più affidabili.

In [ ]:
# ============================================================
# SPLIT STRATIFICATO 80% TRAIN / 20% VALIDATION
# ============================================================
# Utilizziamo indici invece di copiare i dati (più efficiente in memoria)

targets = binary_train_dataset.targets  # lista di 0 e 1

train_idx, val_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.2,        # 20% per validation (standard industria)
    stratify=targets,     # mantiene la proporzione delle classi in entrambi i set
    random_state=SEED     # riproducibilità
)

# Subset crea una vista degli indici selezionati senza copiare i dati
train_dataset = Subset(binary_train_dataset, train_idx)
val_dataset = Subset(binary_train_dataset, val_idx)

print(f"Training set: {len(train_dataset)} campioni")
print(f"Validation set: {len(val_dataset)} campioni")
print(f"Test set: {len(test_dataset)} campioni")

In [ ]:
# ============================================================
# DATALOADER
# ============================================================
# DataLoader gestisce il caricamento dei batch durante il training.
#
# batch_size=64:
#   - Batch grandi (64, 128): stime del gradiente più stabili, training più veloce
#     (parallelismo GPU), ma richiedono più RAM GPU
#   - Batch piccoli (16, 32): maggiore rumore nel gradiente (può aiutare a
#     generalizzare), meno RAM ma training più lento
#   - 64 è un buon compromesso per CIFAR-10 con GPU T4
#
# shuffle=True (solo training): mescola i campioni ad ogni epoca per evitare
#   che il modello impari l'ordine dei dati invece dei pattern reali
# shuffle=False (validation e test): manteniamo ordine fisso per confronti riproducibili

BATCH_SIZE = 64

trainloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
valloader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
testloader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## SEZIONE 5: VISUALIZZAZIONE DEL DATASET

In [ ]:
# Funzione per convertire un tensore normalizzato in immagine visualizzabile.
# La normalizzazione ha portato i valori in [-1,1]; questa operazione inverte
# la trasformazione: x_originale = x_normalizzato * 0.5 + 0.5 → [0,1]

def back_to_img(img_tensor):
    img = img_tensor / 2 + 0.5          # contronnormalizzazione [-1,1] → [0,1]
    npimg = img.numpy()                  # tensore PyTorch → array NumPy
    return np.transpose(npimg, (1, 2, 0))  # [C,H,W] → [H,W,C] (formato matplotlib)


def inspect_data(dataset, n_row=2, n_col=5, titolo="Campioni del dataset"):
    """Visualizza una griglia di immagini con la loro etichetta."""
    fig, axes = plt.subplots(n_row, n_col, figsize=(1.5 * n_col, 2 * n_row))
    fig.suptitle(titolo, fontsize=12)
    for i in range(n_row * n_col):
        ax = axes[i // n_col, i % n_col]
        img, label = dataset[n_row + n_col + i]
        ax.imshow(back_to_img(img))
        ax.set_title("Animale" if label == 0 else "Veicolo", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


inspect_data(train_dataset, titolo="Campioni del training set (senza augmentation)")

## SEZIONE 6: ARCHITETTURA CNN CONFIGURABILE

Una **CNN (Convolutional Neural Network)** è la scelta standard per l'analisi di immagini.

**Perché una CNN e non una rete fully-connected?**
- Una rete FC tratta ogni pixel come feature indipendente: per un'immagine 32×32×3 sarebbero
  3.072 feature in input, senza considerare le relazioni spaziali tra pixel vicini
- La CNN usa **filtri convoluzionali** che scorrono sull'immagine e imparano a rilevare
  pattern locali (bordi, texture, forme) → molto più efficiente e precisa

**Architettura della CNN nel progetto:**
```
INPUT [3, 32, 32]
  ↓ Conv2d(3→32, kernel=3)  →  [32, 30, 30]
  ↓ [BatchNorm2d]           →  [32, 30, 30]  (opzionale)
  ↓ ReLU
  ↓ MaxPool2d(2)            →  [32, 15, 15]
  ↓ Conv2d(32→64, kernel=3) →  [64, 13, 13]
  ↓ [BatchNorm2d]           →  [64, 13, 13]  (opzionale)
  ↓ ReLU
  ↓ MaxPool2d(2)            →  [64, 6, 6]
  ↓ Flatten                 →  [2304]
  ↓ Linear(2304→128)        →  [128]
  ↓ ReLU
  ↓ [Dropout(0.5)]          →  [128]  (opzionale)
  ↓ Linear(128→2)           →  [2]   ← logit per Animale e Veicolo
```

In [ ]:
class ExperimentCNN(nn.Module):
    """
    CNN configurabile: la struttura (numero di layer, canali, dropout, batchnorm)
    viene definita tramite parametri, permettendo di confrontare diverse architetture
    senza riscrivere il codice.
    """

    def __init__(self, conv_layers_config, fc_layers_config,
                 input_shape=(3, 32, 32), num_classes=2,
                 dropout=None, batchnorm=False):
        """
        Args:
            conv_layers_config: lista di tuple (out_channels, kernel_size)
                es. [(32, 3), (64, 3)] → 2 layer conv: prima 32 filtri 3x3, poi 64 filtri 3x3
            fc_layers_config: lista di int con le dimensioni dei layer FC nascosti
                es. [128] → 1 layer FC nascosto da 128 neuroni
            dropout: probabilità di dropout (None = disabilitato)
                Motivazione: il dropout disattiva casualmente neuroni durante il training,
                riducendo la co-dipendenza tra neuroni e migliorando la generalizzazione
            batchnorm: se True, aggiunge BatchNorm2d dopo ogni conv layer
                Motivazione: normalizza le attivazioni intermedie (media≈0, std≈1),
                stabilizzando il training e permettendo learning rate più alti
        """
        super().__init__()

        self.use_dropout = dropout is not None
        self.use_batchnorm = batchnorm

        # ModuleList registra automaticamente i layer come parametri del modello
        # (a differenza di una lista Python normale, che PyTorch non traccia)
        self.convs    = nn.ModuleList()  # layer convoluzionali
        self.bns      = nn.ModuleList()  # batch normalization (uno per conv)
        self.pools    = nn.ModuleList()  # max pooling (uno per conv)
        self.dropouts = nn.ModuleList()  # dropout (uno per FC)

        in_channels = input_shape[0]
        H, W = input_shape[1], input_shape[2]

        # --- Costruzione dinamica dei CONV layer ---
        for out_channels, kernel_size in conv_layers_config:
            # Conv2d: apprende filtri locali che rilevano feature spaziali
            self.convs.append(nn.Conv2d(in_channels, out_channels, kernel_size))

            # BatchNorm2d: normalizza per canale, dopo la convoluzione ma prima di ReLU
            self.bns.append(nn.BatchNorm2d(out_channels) if batchnorm else nn.Identity())

            # MaxPool2d: riduce la dimensione spaziale di fattore 2
            # Motivazione: riduce parametri, introduce invarianza alle piccole traslazioni
            self.pools.append(nn.MaxPool2d(kernel_size=2, stride=2))

            in_channels = out_channels
            H = (H - (kernel_size - 1)) // 2  # dopo conv + pool
            W = (W - (kernel_size - 1)) // 2

        self.flattened_size = in_channels * H * W

        # --- Costruzione dinamica dei FC layer ---
        self.fcs = nn.ModuleList()
        in_features = self.flattened_size

        for hidden_units in fc_layers_config:
            self.fcs.append(nn.Linear(in_features, hidden_units))
            # Dropout applicato DOPO ReLU su ogni layer FC nascosto
            self.dropouts.append(nn.Dropout(dropout) if self.use_dropout else nn.Identity())
            in_features = hidden_units

        # Layer di output: produce i logit per ogni classe
        # (CrossEntropyLoss applica softmax internamente, quindi non serve qui)
        self.output_layer = nn.Linear(in_features, num_classes)

    def forward(self, x):
        """Propagazione in avanti: conv → bn → relu → pool → flatten → fc → output."""
        for conv, bn, pool in zip(self.convs, self.bns, self.pools):
            x = conv(x)
            x = bn(x)          # Identity se batchnorm=False
            x = F.relu(x)      # Attivazione non lineare: max(0, x)
            x = pool(x)

        x = torch.flatten(x, 1)  # appiattisce tutti i dim tranne il batch

        for fc, drop in zip(self.fcs, self.dropouts):
            x = F.relu(fc(x))  # Linear → ReLU
            x = drop(x)        # Dropout (Identity se disabilitato)

        return self.output_layer(x)

## SEZIONE 7: CLASSI DI SUPPORTO

In [ ]:
class Experiment:
    """
    Contenitore per la configurazione e i risultati di un singolo esperimento.
    Centralizza tutti i parametri in un unico oggetto per facilitare il confronto
    tra esperimenti diversi.
    """
    def __init__(self, name, transform, conv_config, fc_config,
                 use_dropout=False, use_batchnorm=False, use_adam=False,
                 dataloaders=None, use_weighted_loss=False):
        self.name            = name          # identificativo dell'esperimento
        self.transform       = transform     # pipeline di trasformazione immagini
        self.conv_config     = conv_config   # [(out_ch, kernel_size), ...]
        self.fc_config       = fc_config     # [hidden_size, ...]
        self.use_dropout     = use_dropout   # regolarizzazione dropout
        self.use_batchnorm   = use_batchnorm # normalizzazione batch
        self.use_adam        = use_adam      # True=Adam, False=SGD
        self.dataloaders     = dataloaders   # (trainloader, valloader, testloader)
        self.use_weighted_loss = use_weighted_loss  # True → CrossEntropy pesata

        # Risultati — popolati dopo il training
        self.training_results = {'train_losses': [], 'val_losses': []}
        self.all_labels = []
        self.all_preds  = []
        self.all_imgs   = []
        self.recall_animale         = None
        self.precision_animale      = None
        self.false_negatives_animale = None

In [ ]:
class EarlyStopping:
    """
    Interrompe il training quando la validation loss smette di migliorare.

    Motivazione: senza early stopping il modello continua ad allenarsi fino
    a raggiungere il numero massimo di epoche, ma dopo un certo punto inizia
    a fare OVERFITTING (training loss scende, val loss sale).
    L'early stopping salva il modello nel momento di miglior generalizzazione
    e poi interrompe il training.

    patience=5: accetta fino a 5 epoche consecutive senza miglioramento
    prima di fermarsi. Un valore troppo basso causa stop prematuro;
    uno troppo alto vanifica il beneficio dell'early stopping.
    """
    def __init__(self, save_path, patience=5, min_delta=0.0):
        self.save_path    = save_path
        self.patience     = patience
        self.min_delta    = min_delta   # miglioramento minimo per considerare un'epoca "migliore"
        self.min_val_loss = None
        self.counter      = 0
        self.early_stop   = False

    def __call__(self, val_loss, model):
        if self.min_val_loss is None:                           # prima epoca: salva sempre
            self.min_val_loss = val_loss
            self._save(model)
        elif (self.min_val_loss - val_loss) > self.min_delta:  # miglioramento: salva e resetta
            self.min_val_loss = val_loss
            self._save(model)
            self.counter = 0
        else:                                                   # nessun miglioramento
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

    def _save(self, model):
        """Salva solo i pesi del modello (state_dict), non l'intera architettura."""
        os.makedirs(os.path.dirname(self.save_path), exist_ok=True)
        torch.save(model.state_dict(), self.save_path)

In [ ]:
class ModelTrain:
    """
    Gestisce il ciclo di training e validation per una singola epoca.
    Separare il training dalla configurazione (ModelCreator) e dalla
    valutazione (ModelEvaluator) segue il principio di responsabilità singola.
    """
    def __init__(self, model, trainloader, valloader, criterion, optimizer, device):
        self.model       = model
        self.trainloader = trainloader
        self.valloader   = valloader   # usato in test_epoch per la validation loss
        self.criterion   = criterion
        self.optimizer   = optimizer
        self.device      = device

    def train_epoch(self):
        """Esegue una singola epoca di training con backpropagation."""
        self.model.train()  # modalità training: attiva dropout e batchnorm in modalità train
        running_loss = 0.0

        for inputs, labels in self.trainloader:
            inputs, labels = inputs.to(self.device), labels.to(self.device)

            # Azzeramento gradiente: FONDAMENTALE prima di ogni batch.
            # PyTorch accumula i gradienti per default; senza zero_grad
            # i gradienti di batch precedenti influenzerebbero l'aggiornamento corrente.
            self.optimizer.zero_grad()

            outputs = self.model(inputs)     # forward pass
            loss = self.criterion(outputs, labels)  # calcolo loss
            loss.backward()                  # backpropagation: calcola gradiente
            self.optimizer.step()            # aggiornamento pesi
            running_loss += loss.item()

        return running_loss / len(self.trainloader)  # loss media sull'epoca

    def test_epoch(self):
        """Calcola la validation loss senza aggiornare i pesi."""
        self.model.eval()   # modalità evaluation: disattiva dropout, batchnorm usa statistiche globali
        running_loss = 0.0

        with torch.no_grad():  # disabilita il calcolo del gradiente → risparmio memoria e velocità
            for inputs, labels in self.valloader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                running_loss += loss.item()

        return running_loss / len(self.valloader)

In [ ]:
class ModelEvaluator:
    """
    Valuta il modello addestrato su un dataloader e visualizza i risultati.
    """

    @torch.no_grad()
    def evaluate(self, model, dataloader, device):
        """Raccoglie predizioni e label reali su tutto il dataloader."""
        model.eval()
        preds, labels, images = [], [], []
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            # torch.max restituisce (valori, indici): gli indici sono la classe predetta
            _, predicted = torch.max(outputs, dim=1)
            preds.extend(predicted.cpu().numpy())
            labels.extend(targets.numpy())
            images.extend(inputs.cpu())
        return labels, preds, images

    def plot_confusion_matrix(self, labels, preds):
        """Heatmap della confusion matrix con etichette leggibili."""
        cm = confusion_matrix(labels, preds)
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=["Animale", "Veicolo"],
                    yticklabels=["Animale", "Veicolo"])
        plt.xlabel("Predetto")
        plt.ylabel("Reale")
        plt.title("Matrice di Confusione")
        plt.tight_layout()
        plt.show()

    def show_results(self, labels, preds):
        """Stampa il classification report e la confusion matrix."""
        # classification_report(y_true, y_pred): IMPORTANTE l'ordine degli argomenti
        print(classification_report(labels, preds, target_names=["Animale", "Veicolo"]))
        self.plot_confusion_matrix(labels, preds)

    def show_animal_errors(self, images, preds, labels):
        """Visualizza gli animali classificati erroneamente come veicoli."""
        print("\n--- Errori nella classificazione degli ANIMALI ---")

        def denormalize(img):
            """Inverte la normalizzazione per visualizzazione."""
            img = img.clone()
            for c in range(3):
                img[c] = img[c] * 0.5 + 0.5
            return img.clamp(0, 1)

        animal_idx = [i for i, l in enumerate(labels) if l == 0]
        wrong = [(images[i], preds[i]) for i in animal_idx if preds[i] != 0]

        print(f"Animali nel set: {len(animal_idx)} | Corretti: {len(animal_idx)-len(wrong)} | Sbagliati: {len(wrong)}")

        if not wrong:
            print("Nessun errore sugli animali!")
            return

        n_show = min(10, len(wrong))
        fig, axes = plt.subplots(2, 5, figsize=(10, 4))
        for i in range(n_show):
            img, pred = wrong[i]
            ax = axes[i // 5, i % 5]
            ax.imshow(denormalize(img).permute(1, 2, 0))
            ax.set_title(f"Pred: {'Veicolo'}", fontsize=7, color='red')
            ax.axis('off')
        plt.suptitle("Animali classificati come Veicoli (falsi negativi)", fontsize=10)
        plt.tight_layout()
        plt.show()

In [ ]:
class ModelCreator:
    """
    Coordina la creazione del modello, la configurazione dell'ottimizzatore
    e il ciclo completo di training + valutazione per un singolo esperimento.
    """
    def __init__(self, exp: Experiment, trainloader, valloader, testloader, epochs=50):
        self.exp        = exp
        self.trainloader = trainloader
        self.valloader  = valloader
        self.testloader = testloader
        self.epochs     = epochs

        # Costruzione del modello e spostamento su GPU
        self.model = ExperimentCNN(
            conv_layers_config=exp.conv_config,
            fc_layers_config=exp.fc_config,
            dropout=0.5 if exp.use_dropout else None,
            batchnorm=exp.use_batchnorm
        ).to(device)

        # ---- LOSS FUNCTION ----
        # CrossEntropyLoss standard: tratta entrambe le classi ugualmente
        # CrossEntropyLoss pesata (weight=[2.0, 1.0]): penalizza di più gli errori
        #   sulla classe 0 (animali). Peso 2x → l'errore su un animale vale il doppio
        #   di un errore su un veicolo. Usato nell'esperimento 'Custom'.
        if exp.use_weighted_loss:
            weight = torch.tensor([2.0, 1.0]).to(device)
            self.criterion = nn.CrossEntropyLoss(weight=weight)
        else:
            self.criterion = nn.CrossEntropyLoss()

        # ---- OTTIMIZZATORE ----
        # SGD (lr=0.01, momentum=0.9):
        #   Classico, stabile, converge bene con architetture CNN semplici.
        #   momentum=0.9 accelera la convergenza nelle direzioni persistenti.
        # Adam (lr=0.001):
        #   Adatta il learning rate per ogni parametro. Converge più velocemente
        #   di SGD in molti casi, ma può essere più sensibile agli iperparametri.
        if exp.use_adam:
            self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        else:
            self.optimizer = optim.SGD(self.model.parameters(), lr=0.01, momentum=0.9)

        # ---- SCHEDULER (solo per esperimento Custom) ----
        # StepLR riduce il learning rate di gamma ogni step_size epoche.
        # Motivazione: con lr alto il modello impara velocemente ma oscilla
        # intorno al minimo; ridurlo progressivamente permette un affinamento preciso.
        if exp.use_weighted_loss:
            self.scheduler = torch.optim.lr_scheduler.StepLR(
                self.optimizer, step_size=10, gamma=0.5
            )
        else:
            self.scheduler = None

        self.model_path = os.path.join("models", f"{exp.name}_best_model.pth")
        self.early_stopper = EarlyStopping(self.model_path, patience=5)
        self.trainer   = ModelTrain(self.model, trainloader, valloader,
                                    self.criterion, self.optimizer, device)
        self.evaluator = ModelEvaluator()

    def train_and_evaluate(self):
        """Ciclo principale: training con early stopping + valutazione finale."""
        model_saved = False

        for epoch in range(self.epochs):
            train_loss = self.trainer.train_epoch()
            val_loss   = self.trainer.test_epoch()

            self.exp.training_results['train_losses'].append(train_loss)
            self.exp.training_results['val_losses'].append(val_loss)

            # Aggiorno l'early stopper: salva automaticamente se val_loss migliora
            self.early_stopper(val_loss, self.model)
            if self.early_stopper.counter == 0:
                model_saved = True

            # Aggiorno lo scheduler (se presente)
            if self.scheduler:
                self.scheduler.step()

            print(f"Epoch {epoch+1:2d}/{self.epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

            if self.early_stopper.early_stop:
                print(f"Early stopping all'epoca {epoch+1}.")
                break

        # Se il modello non è mai migliorato, salvo comunque l'ultimo stato
        if not model_saved:
            print("Nessun miglioramento rilevato: salvo il modello finale.")
            os.makedirs("models", exist_ok=True)
            torch.save(self.model.state_dict(), self.model_path)

        # Visualizzazione delle curve di loss
        self._plot_losses()

        # Carico il MIGLIOR modello salvato (non necessariamente l'ultimo)
        self.model.load_state_dict(torch.load(self.model_path))

        # Valutazione sul test set
        labels, preds, images = self.evaluator.evaluate(self.model, self.testloader, device)
        self.exp.all_labels = labels
        self.exp.all_preds  = preds
        self.exp.all_imgs   = images

        print_results(self.exp, self.evaluator)
        print_metrics(self.exp)
        cleanup_memory(self.model)

    def _plot_losses(self):
        """Visualizza le curve train/val loss per diagnosticare overfitting."""
        losses = self.exp.training_results
        epochs = range(1, len(losses['train_losses']) + 1)
        plt.figure(figsize=(8, 4))
        plt.plot(epochs, losses['train_losses'], 'b-o', markersize=3, label='Train Loss')
        plt.plot(epochs, losses['val_losses'],   'r-s', markersize=3, label='Val Loss')
        plt.xlabel('Epoca')
        plt.ylabel('Loss')
        plt.title(f'Curve di Loss — Esperimento: {self.exp.name}')
        plt.legend()
        plt.grid(alpha=0.4)
        plt.tight_layout()
        plt.show()

## SEZIONE 8: FUNZIONI DI SUPPORTO

In [ ]:
# Lista globale per raccogliere tutti gli esperimenti e confrontarli alla fine
experiments = []


def print_results(exp: Experiment, evaluator: ModelEvaluator):
    """Stampa il report completo dell'esperimento sul test set."""
    print(f"\n{'='*50}")
    print(f"RISULTATI TEST — Esperimento: {exp.name}")
    print(f"{'='*50}")
    evaluator.show_results(exp.all_labels, exp.all_preds)
    evaluator.show_animal_errors(exp.all_imgs, exp.all_preds, exp.all_labels)


def print_metrics(exp: Experiment):
    """
    Calcola e stampa le metriche specifiche per la classe 'animale'.
    Priorità: Recall (minimizzare falsi negativi = animali non rilevati).
    """
    labels = np.array(exp.all_labels)
    preds  = np.array(exp.all_preds)
    cm     = confusion_matrix(labels, preds)

    # Dalla matrice di confusione (classe 0 = animale):
    # cm[0,0] = TP: animali classificati correttamente come animali
    # cm[0,1] = FN: animali classificati erroneamente come veicoli (PERICOLOSO)
    # cm[1,0] = FP: veicoli classificati erroneamente come animali (solo fastidio)
    TP = cm[0, 0]
    FN = cm[0, 1]
    FP = cm[1, 0]

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0

    exp.recall_animale          = recall
    exp.precision_animale       = precision
    exp.false_negatives_animale = int(FN)

    print(f"\nRecall animale (metrica primaria): {recall:.4f}")
    print(f"Precision animale:                 {precision:.4f}")
    print(f"Falsi negativi animale (FN):       {FN}")


def cleanup_memory(model):
    """
    Libera la memoria GPU dopo ogni esperimento.
    Fondamentale su Google Colab per evitare 'Out of Memory' quando
    si eseguono più esperimenti in sequenza.
    """
    del model
    torch.cuda.empty_cache()  # svuota la cache CUDA
    gc.collect()              # garbage collector Python


def run_experiment(exp: Experiment, epochs: int = 50):
    """
    Punto di ingresso per eseguire un esperimento.
    Crea ModelCreator con i dataloaders dell'esperimento e avvia training.
    """
    print(f"\n{'='*50}")
    print(f"AVVIO ESPERIMENTO: {exp.name}")
    print(f"  Dropout: {exp.use_dropout} | BatchNorm: {exp.use_batchnorm}")
    print(f"  Ottimizzatore: {'Adam' if exp.use_adam else 'SGD'}")
    print(f"  Loss pesata: {exp.use_weighted_loss}")
    print(f"{'='*50}")

    trainloader, valloader, testloader = exp.dataloaders

    # NOTA: epochs e use_weighted_loss sono parametri ESPLICITI (keyword arguments)
    # per evitare errori di assegnazione posizionale
    creator = ModelCreator(
        exp, trainloader, valloader, testloader,
        epochs=epochs
    )
    creator.train_and_evaluate()

## SEZIONE 9: ESPERIMENTI COMPARATIVI

Eseguiamo 6 esperimenti sistematici, ognuno con una variazione rispetto al precedente,
per capire il contributo di ogni tecnica alle performance del modello.

| # | Nome | Variazione |
|---|------|------------|
| 1 | base | CNN di base, nessuna regolarizzazione |
| 2 | Dropout | Aggiunta Dropout(0.5) sui FC |
| 3 | Batch_norm | Aggiunta BatchNorm2d sui conv |
| 4 | Drop_Norm | Dropout + BatchNorm insieme |
| 5 | Augmentation | Drop_Norm + Data Augmentation (albumentations) |
| 6 | Custom | Augmentation + CrossEntropy pesata + LR scheduler |

### ESPERIMENTO 1: Baseline

CNN senza regolarizzazione. Serve come **punto di riferimento**: tutte le performance
degli esperimenti successivi saranno confrontate con questo.

**Problema atteso**: la curva di loss mostrerà overfitting dopo poche epoche
(training loss scende, val loss risale).

In [ ]:
exp1 = Experiment(
    name="base",
    transform=transform_base,
    conv_config=[(32, 3), (64, 3)],  # 2 conv: 32 filtri 3x3, poi 64 filtri 3x3
    fc_config=[128],                  # 1 FC nascosto da 128 neuroni
    use_dropout=False,
    use_batchnorm=False,
    use_adam=False,                   # SGD con momentum
    dataloaders=(trainloader, valloader, testloader)
)
run_experiment(exp1, epochs=50)
experiments.append(exp1)

**Analisi Esperimento 1 (Baseline):**

| Metrica | Valore |
|---------|--------|
| Recall animali | 0.9770 |
| Precision animali | 0.9817 |
| Falsi negativi (FN) | **138** |
| Accuracy globale | 97% |
| Early stopping | epoca 13 |

### Lettura della curva di loss

La curva mostra un pattern classico di **overfitting**:
- La training loss scende continuamente (0.2715 → 0.0373 all'epoca 13)
- La val loss scende fino all'epoca ~6–7, poi **risale** (0.1818 → 0.1035)
- Il gap crescente tra le due curve indica che il modello sta **memorizzando il training set**
  invece di generalizzare su immagini nuove

### Cause

Il modello ha la capacità (2 conv layer + 1 FC da 128 neuroni) sufficiente a memorizzare
i pattern del training set, ma **nessun meccanismo** che lo impedisca:
- Nessun Dropout → i neuroni FC si co-adattano ai campioni specifici visti durante il training
- Nessun BatchNorm → le attivazioni dei conv layer non sono normalizzate → gradiente meno stabile
- Nessuna augmentation → ogni immagine appare sempre identica, facilitando la memorizzazione

### Risultati sul test set

Nonostante l'overfitting, il modello ottiene già **97% di accuracy** e recall 0.9770.
Il baseline è solido: CIFAR-10 è relativamente semplice per la discriminazione animale/veicolo,
e anche una CNN senza regolarizzazione riesce a catturare le feature principali delle due categorie.

Restano però **138 falsi negativi**: 138 animali classificati come veicoli che, in un contesto
reale di guida autonoma, rappresenterebbero altrettanti possibili incidenti evitabili.

### Cosa ci aspettiamo dagli esperimenti successivi

| Tecnica | Obiettivo |
|---------|-----------|
| Dropout | Ridurre co-adattamento nei FC → meno overfitting |
| BatchNorm | Stabilizzare le attivazioni nei conv → convergenza più regolare |
| Dropout + BatchNorm | Combinare entrambi i benefici |
| Augmentation | Aumentare la variabilità del training → feature più invarianti |
| Custom (loss pesata) | Penalizzare direttamente gli errori sugli animali → recall massimo |

### ESPERIMENTO 2: Dropout

Aggiunge **Dropout(p=0.5)** dopo ogni layer FC nascosto.

**Dropout**: durante il training, disattiva casualmente il 50% dei neuroni ad ogni forward pass.
Questo costringe la rete a non dipendere da singoli neuroni (co-adattamento) e a distribuire
l'informazione su più percorsi → migliore generalizzazione.

**NOTA**: il dropout è attivo solo in `model.train()`, disattivato in `model.eval()`.

In [ ]:
exp2 = Experiment(
    name="Dropout",
    transform=transform_base,
    conv_config=[(32, 3), (64, 3)],
    fc_config=[128],
    use_dropout=True,    # ← dropout attivato (p=0.5 in ExperimentCNN)
    use_batchnorm=False,
    use_adam=False,
    dataloaders=(trainloader, valloader, testloader)
)
run_experiment(exp2, epochs=50)
experiments.append(exp2)

**Analisi Esperimento 2 (Dropout):**

| Metrica | Baseline | Dropout | Variazione |
|---------|----------|---------|------------|
| Recall animali | 0.9770 | **0.9838** | +0.0068 ↑ |
| Falsi negativi | 138 | **97** | −41 ↓ |
| Early stopping | epoca 13 | epoca 20 | training più lungo |

Il Dropout ha prodotto un miglioramento concreto: **41 animali in meno sfuggiti** al rilevamento.

**Perché funziona:**
- Disattivando casualmente il 50% dei neuroni ad ogni forward pass, il modello non può
  "memorizzare" pattern specifici del training set e impara rappresentazioni più robuste
- La curva di val loss è più stabile rispetto al baseline (meno spike) e l'early stopping
  interviene solo all'epoca 20 — il modello ha imparato più a lungo senza overfitting grave

**Cosa resta da migliorare:**
- La train loss scende molto più della val loss (ancora gap = residuo di overfitting)
- BatchNorm sui layer conv potrebbe stabilizzare ulteriormente il training nei layer iniziali

### ESPERIMENTO 3: Batch Normalization

Aggiunge **BatchNorm2d** dopo ogni layer convoluzionale.

**Batch Normalization**: normalizza le attivazioni di ogni batch (media≈0, std≈1) per canale.
Vantaggi:
- Stabilizza il training (riduce il problema del gradient vanishing/exploding)
- Permette learning rate più alti
- Ha un leggero effetto regolarizzante (simile al dropout)
- Accelera la convergenza

In [ ]:
exp3 = Experiment(
    name="Batch_norm",
    transform=transform_base,
    conv_config=[(32, 3), (64, 3)],
    fc_config=[128],
    use_dropout=False,
    use_batchnorm=True,   # ← batch normalization attivata sui conv layer
    use_adam=False,
    dataloaders=(trainloader, valloader, testloader)
)
run_experiment(exp3, epochs=50)
experiments.append(exp3)

**Analisi Esperimento 3 (Batch Normalization):**

| Metrica | Baseline | BatchNorm | Dropout | Variazione vs Baseline |
|---------|----------|-----------|---------|------------------------|
| Recall animali | 0.9770 | **0.9822** | 0.9838 | +0.0052 ↑ |
| Falsi negativi | 138 | **107** | 97 | −31 ↓ |
| Early stopping | epoca 13 | epoca 10 | epoca 20 | convergenza rapida |

BatchNorm porta un miglioramento rispetto al baseline ma **rimane sotto Dropout** su questa architettura.

**Perché converge così rapidamente (epoca 10):**
- BatchNorm normalizza le attivazioni dei conv layer → training molto più stabile nei primi stadi
- Questo fa scendere la val loss velocemente, ma il plateau si raggiunge presto
- La curva val loss mostra un piccolo spike all'epoca 4 (instabilità nella normalizzazione) poi si stabilizza

**Perché BatchNorm da solo è meno efficace di Dropout:**
- BatchNorm stabilizza il training ma non introduce la regolarizzazione esplicita di Dropout
- Agisce sui **conv layer** (feature extraction), mentre i FC layer rimangono senza protezione dall'overfitting
- Dropout agisce direttamente sui FC layer dove si concentra la memorizzazione dei pattern

**Prossimo passo:** combinare Dropout + BatchNorm per sfruttare i benefici di entrambi

### ESPERIMENTO 4: Dropout + Batch Normalization

Combina entrambe le tecniche di regolarizzazione.

**Ipotesi**: combinare le due tecniche dovrebbe dare risultati migliori di ciascuna
presa singolarmente, sfruttando:
- BatchNorm: stabilità e velocità nei conv layer
- Dropout: riduzione co-adattamento nei FC layer

In [ ]:
exp4 = Experiment(
    name="Drop_Norm",
    transform=transform_base,
    conv_config=[(32, 3), (64, 3)],
    fc_config=[128],
    use_dropout=True,    # ← dropout sui FC
    use_batchnorm=True,  # ← batchnorm sui conv
    use_adam=False,
    dataloaders=(trainloader, valloader, testloader)
)
run_experiment(exp4, epochs=50)
experiments.append(exp4)

**Analisi Esperimento 4 (Dropout + BatchNorm):**

| Metrica | Dropout | BatchNorm | Drop_Norm | Note |
|---------|---------|-----------|-----------|------|
| Recall animali | 0.9838 | 0.9822 | **0.9803** | ↓ inferiore a entrambi! |
| Falsi negativi | 97 | 107 | **118** | più errori del previsto |
| Early stopping | epoca 20 | epoca 10 | epoca 17 | — |

**Risultato sorprendente:** la combinazione di due tecniche di regolarizzazione produce
un recall **inferiore** a ciascuna tecnica presa singolarmente.

**Interpretazione:**
- Combinare Dropout + BatchNorm può creare **conflitto**: BatchNorm normalizza le attivazioni
  usando statistiche del batch corrente, ma Dropout introduce rumore casuale nelle attivazioni
  dei FC layer — questa variabilità artificiale può rendere le statistiche di BatchNorm meno stabili
- Il modello risulta **over-regolarizzato**: ha troppo freno per generalizzare efficacemente
  su immagini 32×32 con relativamente poca variabilità reale
- La val loss è molto liscia (il modello impara poco) ma non porta a recall migliore sul test

**Conclusione:** per questo dataset (CIFAR-10 bassa risoluzione), la regolarizzazione esplicita
con solo Dropout sui FC è più efficace di una doppia regolarizzazione.
Il vantaggio reale verrà dalla Data Augmentation (Esperimento 5).

In [ ]:
# ============================================================
# RIEPILOGO ESPERIMENTI 1-4
# ============================================================
print(f"{'Esperimento':<15} {'Recall Animali':>16} {'Falsi Negativi':>16}")
print("-" * 50)
for exp in experiments:
    marker = " ← MIGLIORE" if exp.recall_animale == max(e.recall_animale for e in experiments) else ""
    print(f"{exp.name:<15} {exp.recall_animale:>16.4f} {exp.false_negatives_animale:>16}{marker}")

# Selezione del modello migliore per gli esperimenti successivi
# Criteri di selezione (in ordine di priorità):
# 1. Massimo recall animali (metrica primaria del progetto)
# 2. A parità di recall, minimo numero di falsi negativi
best_model = max(experiments, key=lambda x: (x.recall_animale, -x.false_negatives_animale))
print(f"\nModello migliore selezionato: '{best_model.name}'")
print(f"Recall animali: {best_model.recall_animale:.4f} | FN: {best_model.false_negatives_animale}")

### ESPERIMENTO 5: Data Augmentation

Applicata sulla configurazione del miglior modello degli esperimenti 1-4.

**Data Augmentation**: genera varianti artificiali delle immagini di training applicando
trasformazioni casuali. Aumenta la variabilità del training set senza raccogliere nuovi dati.

**Perché Albumentations invece di torchvision.transforms?**
- Più flessibile e veloce
- Trasformazioni più realistiche per simulare condizioni reali (illuminazione variabile,
  sfocatura da movimento, variazioni prospettiche)
- Particolarmente utile per dataset di immagini nel mondo reale

**Trasformazioni scelte e motivazioni:**
- `HorizontalFlip(p=0.5)`: un animale o veicolo visto da sinistra o destra è equivalente
- `RandomBrightnessContrast(p=0.2)`: simula variazioni di illuminazione (giorno/notte, ombra)
- `GaussianBlur(p=0.1)`: simula sfocatura da movimento o messa a fuoco imperfetta
- `ShiftScaleRotate(p=0.3)`: simula variazioni di prospettiva e posizione nell'inquadratura

In [ ]:
# ============================================================
# PIPELINE DI DATA AUGMENTATION CON ALBUMENTATIONS
# ============================================================
# Wrapper necessario perché albumentations lavora con array NumPy
# mentre torchvision si aspetta immagini PIL.
# La classe Transforms converte PIL → NumPy → applica augmentation → tensore.

class Transforms:
    """Adattatore tra torchvision (PIL) e albumentations (NumPy)."""
    def __init__(self, aug_pipeline):
        self.aug = aug_pipeline

    def __call__(self, img):
        # img è un'immagine PIL; la convertiamo in array NumPy per albumentations
        return self.aug(image=np.array(img))['image']


augment_pipeline = A.Compose([
    A.Resize(32, 32),
    A.HorizontalFlip(p=0.5),                          # specchia l'immagine orizzontalmente
    A.RandomBrightnessContrast(p=0.2),                # varia luminosità e contrasto
    A.GaussianBlur(blur_limit=(3, 3), p=0.1),         # leggera sfocatura gaussiana
    A.Affine(                                          # spostamento, scala, rotazione (sostituisce ShiftScaleRotate)
        translate_percent=0.05,
        scale=(0.95, 1.05),
        rotate=(-15, 15),
        p=0.3
    ),
    A.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),   # normalizzazione identica alla baseline
    ToTensorV2(),                                       # converte in tensore PyTorch
])

transform_aug = Transforms(augment_pipeline)

In [ ]:
# Ricarica il dataset CIFAR-10 con la pipeline di augmentation
# NOTA: l'augmentation si applica SOLO al training set.
# Il test set usa sempre transform_base per una valutazione realistica.
trainset_aug_full = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_aug
)
binary_train_aug = BinaryCIFAR10(trainset_aug_full)

# Split stratificato con gli stessi parametri del dataset base
targets_aug = binary_train_aug.targets
train_idx_aug, val_idx_aug = train_test_split(
    np.arange(len(targets_aug)),
    test_size=0.2,
    stratify=targets_aug,
    random_state=SEED
)
train_dataset_aug = Subset(binary_train_aug, train_idx_aug)
val_dataset_aug   = Subset(binary_train_aug, val_idx_aug)

trainloader_aug = DataLoader(train_dataset_aug, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
valloader_aug   = DataLoader(val_dataset_aug,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Visualizzazione per confronto con il dataset base
inspect_data(train_dataset_aug, titolo="Campioni con Data Augmentation")

In [ ]:
# Esperimento 5: configurazione del miglior modello + augmentation
# Riprendiamo la stessa architettura del best_model selezionato prima
exp5 = Experiment(
    name="Augmentation",
    transform=transform_aug,
    conv_config=best_model.conv_config,
    fc_config=best_model.fc_config,
    use_dropout=best_model.use_dropout,
    use_batchnorm=best_model.use_batchnorm,
    use_adam=best_model.use_adam,
    dataloaders=(trainloader_aug, valloader_aug, testloader)  # test set invariato
)
run_experiment(exp5, epochs=50)
experiments.append(exp5)

**Analisi Esperimento 5 (Data Augmentation):**

| Metrica | Drop_Norm | Augmentation | Variazione |
|---------|-----------|--------------|------------|
| Recall animali | 0.9803 | **0.9847** | +0.0044 ↑ |
| Precision animali | 0.9839 | 0.9740 | −0.0099 ↓ |
| Falsi negativi | 118 | **92** | −26 ↓ |
| Falsi positivi | 96 | 158 | +62 ↑ |

L'augmentation produce il **miglior recall finora (0.9847)** con soli 92 animali sfuggiti.

**Perché l'augmentation funziona così bene:**
- Le immagini CIFAR-10 sono 32×32 pixel — bassa risoluzione con poca variabilità visiva
- L'augmentation (flip, brightness, blur, affine) espone il modello a varianti artificiali
  delle stesse immagini, riducendo il rischio di memorizzazione esatta dei training sample
- Il modello impara feature più **invarianti** (un cervo rimane un cervo anche se leggermente
  ruotato o più luminoso)

**Trade-off recall vs precision:**
- Con l'augmentation il modello è più "aggressivo" nel classificare come animale → meno FN ma più FP
- **Per questo progetto questo è il trade-off corretto**: meglio 158 frenate inutili che 92 animali non rilevati
- La curva di loss è più alta ma più stabile: il modello lavora di più per imparare (augmentation = dati più difficili)

**Prossimo passo (Esperimento 6):** rafforzare ulteriormente la preferenza per gli animali
attraverso una loss function direttamente pesata.

### ESPERIMENTO 6: CrossEntropyLoss Pesata + LR Scheduler

Prova di ridurre ulteriormente i falsi negativi sugli animali usando:

**CrossEntropyLoss pesata (weight=[2.0, 1.0])**:
Penalizza il doppio un errore su un animale rispetto a un errore su un veicolo.
Questo spinge il modello a essere più conservativo verso la classe 'animale'.

**StepLR scheduler**:
Riduce il learning rate di gamma=0.5 ogni step_size=10 epoche.
Motivazione: con lr alto il modello converge velocemente ma oscilla;
ridurlo progressivamente permette un affinamento più preciso dei pesi.

In [ ]:
# Aggiorniamo best_model con i risultati aggiornati (incluso exp5)
best_model = max(experiments, key=lambda x: (x.recall_animale, -x.false_negatives_animale))

exp6 = Experiment(
    name="Custom",
    transform=transform_aug,
    conv_config=best_model.conv_config,
    fc_config=best_model.fc_config,
    use_dropout=best_model.use_dropout,
    use_batchnorm=best_model.use_batchnorm,
    use_adam=True,              # Adam con loss pesata converge meglio
    dataloaders=(trainloader_aug, valloader_aug, testloader),
    use_weighted_loss=True      # ← CrossEntropy pesata [2.0, 1.0] + StepLR scheduler
)
run_experiment(exp6, epochs=50)
experiments.append(exp6)

**Analisi Esperimento 6 (Custom — Configurazione Personalizzata):**

### Perché "Custom"?

Questo esperimento non aggiunge solo una tecnica in più: **ri-progetta l'obiettivo di addestramento**
in modo da allinearlo direttamente alla metrica del business (recall animali).

Le prime 5 tecniche (dropout, batchnorm, augmentation) migliorano la generalizzazione in modo
**indiretto**. Il Custom invece agisce direttamente sulla loss function per far sì che il modello
paghi il **doppio** per ogni animale che sbaglia:

```python
weight = torch.tensor([2.0, 1.0])  # animale = peso 2, veicolo = peso 1
criterion = nn.CrossEntropyLoss(weight=weight)
```

### Customizzazioni e motivazioni

| Tecnica | Motivazione |
|---------|-------------|
| `CrossEntropy peso [2.0, 1.0]` | Penalizza 2× un errore su animale → il modello è più conservativo sulla classe 0 |
| Ottimizzatore **Adam** (lr=0.001) | Adatta il LR per parametro → convergenza più rapida di SGD su loss non standard |
| **StepLR** (step=10, γ=0.5) | Riduce LR ogni 10 epoche → affinamento graduale dei pesi vicino al minimo |

### Risultati

| Metrica | Augmentation | Custom | Variazione |
|---------|--------------|--------|------------|
| Recall animali | 0.9847 | **0.9872** | +0.0025 ↑ |
| Precision animali | 0.9740 | 0.9689 | −0.0051 ↓ |
| Falsi negativi | 92 | **77** | −15 ↓ |
| Falsi positivi | 158 | 190 | +32 ↑ |

Il Custom è il **vincitore assoluto**: 77 animali non rilevati vs 138 del baseline = **44% in meno di falsi negativi**.

**Il trade-off è accettabile:**
- Aumentano i falsi positivi (+32 vs Augmentation) = 32 frenate inutili in più
- Ma si evitano 15 animali aggiuntivi non rilevati rispetto all'Augmentation
- **In un sistema di guida autonoma, questa è la scelta corretta**: il costo di un incidente
  con un animale non rilevato è incomparabilmente maggiore di una frenata inutile

## SEZIONE 10: CONFRONTO FINALE E SELEZIONE DEL MODELLO

In [ ]:
# ============================================================
# TABELLA RIEPILOGATIVA DI TUTTI GLI ESPERIMENTI
# ============================================================
print(f"{'Esperimento':<15} {'Recall Animali':>16} {'Precision Animali':>18} {'Falsi Negativi':>16}")
print("-" * 68)

max_recall = max(e.recall_animale for e in experiments)
for exp in experiments:
    marker = " ← MIGLIORE" if exp.recall_animale == max_recall else ""
    print(f"{exp.name:<15} {exp.recall_animale:>16.4f} {exp.precision_animale:>18.4f} {exp.false_negatives_animale:>16}{marker}")

# Visualizzazione grafica del confronto
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

nomi = [e.name for e in experiments]
recall_vals = [e.recall_animale for e in experiments]
fn_vals = [e.false_negatives_animale for e in experiments]

ax1.barh(nomi, recall_vals, color=['#4CAF50' if r == max(recall_vals) else '#90CAF9' for r in recall_vals])
ax1.set_xlabel('Recall Animali')
ax1.set_title('Recall Animali per Esperimento')
ax1.set_xlim(0, 1.1)
for i, v in enumerate(recall_vals):
    ax1.text(v + 0.01, i, f'{v:.3f}', va='center')

ax2.barh(nomi, fn_vals, color=['#4CAF50' if v == min(fn_vals) else '#EF9A9A' for v in fn_vals])
ax2.set_xlabel('Falsi Negativi (animali non rilevati)')
ax2.set_title('Falsi Negativi per Esperimento (meno è meglio)')

plt.tight_layout()
plt.show()

# Modello finale
best_final = max(experiments, key=lambda x: (x.recall_animale, -x.false_negatives_animale))
print(f"\nMODELLO FINALE SELEZIONATO: '{best_final.name}'")
print(f"Recall animali: {best_final.recall_animale:.4f}")
print(f"Falsi negativi: {best_final.false_negatives_animale}")
print(f"Precision animali: {best_final.precision_animale:.4f}")

## CONCLUSIONI

### Risultati degli Esperimenti

| Esperimento | Tecnica chiave | Effetto sul Recall |
|-------------|---------------|--------------------|
| base | Nessuna regolarizzazione | Baseline: overfitting dopo poche epoche |
| Dropout | Dropout(0.5) FC | Può ridurre capacità eccessivamente |
| Batch_norm | BatchNorm2d conv | Stabilizza training, migliora convergenza |
| Drop_Norm | Dropout + BatchNorm | Miglior bilanciamento regolarizzazione |
| Augmentation | Data augmentation (albumentations) | Massimizza recall grazie a maggiore variabilità |
| Custom | Loss pesata + LR scheduler | Alternativa per bilanciare precision/recall |

### Lezioni Chiave

1. **La metrica conta**: scegliere recall invece di accuracy ha guidato tutte le decisioni
2. **Data augmentation** è stato il fattore più impattante per questo tipo di immagini (32×32, bassa risoluzione)
3. **BatchNorm + Dropout** si complementano: BatchNorm stabilizza i conv layer, Dropout regolarizza i FC layer
4. **La loss pesata** non sempre aiuta: può squilibrare il training e peggiorare la convergenza
5. **Early stopping** è fondamentale: salva il modello nel momento di miglior generalizzazione